# Evaluacija modela — LSTM vs. BERT

U ovom notebooku evaluiramo istrenirane modele na **test skupu** (~15% podataka) i upoređujemo performanse.

**Metrike:**
- Classification report (precision, recall, F1 po klasi)
- Confusion matrix (heatmap)
- ROC krive (one-vs-rest)
- Primeri pogrešno klasifikovanih tiketa
- Poređenje LSTM vs. BERT

Svi rezultati se čuvaju u `results/` folderu kao CSV i PNG fajlovi.

## 1. Priprema okruženja

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.evaluate import evaluate_all
from src.train import get_device

device = get_device()
print(f"Uređaj: {device}")
print(f"LSTM checkpoint: {config.LSTM_BEST_MODEL_PATH.exists()}")
print(f"BERT checkpoint: {config.BERT_BEST_MODEL_PATH.exists()}")

## 2. Pokretanje evaluacije

Funkcija `evaluate_all()` automatski evaluira dostupne modele. Ako `best_bert.pt` ne postoji, evaluira se samo LSTM.

In [ ]:
results = evaluate_all()
results.keys()

## 3. LSTM — summary metrike

In [ ]:
if results.get("lstm") is not None:
    lstm_summary = pd.read_csv(config.RESULTS_DIR / "lstm" / "summary_metrics.csv")
    lstm_summary
else:
    print("LSTM model nije dostupan za evaluaciju.")

## 4. Classification report (LSTM)

In [ ]:
if results.get("lstm") is not None:
    lstm_report = pd.read_csv(config.RESULTS_DIR / "lstm" / "classification_report.csv")
    lstm_report

## 5. Confusion matrix (LSTM)

Dijagonalni elementi predstavljaju tačne klasifikacije. Van dijagonale su greške modela.

In [ ]:
cm_path = config.RESULTS_DIR / "lstm" / "confusion_matrix.png"
if cm_path.exists():
    display(Image(filename=str(cm_path)))

## 6. ROC krive (LSTM)

In [ ]:
roc_path = config.RESULTS_DIR / "lstm" / "roc_curves.png"
if roc_path.exists():
    display(Image(filename=str(roc_path)))

if results.get("lstm") is not None:
    auc_df = pd.read_csv(config.RESULTS_DIR / "lstm" / "roc_auc.csv")
    auc_df

## 7. Primeri grešaka (LSTM)

Tiketi koje je model pogrešno klasifikovao, sortirani po pouzdanosti predikcije.

In [ ]:
errors_path = config.RESULTS_DIR / "lstm" / "misclassified_examples.csv"
if errors_path.exists():
    errors_df = pd.read_csv(errors_path).head(10)
    for _, row in errors_df.iterrows():
        print("=" * 80)
        print(row["explanation"])
        print("-" * 80)
        print(row["text"])
        print()

## 8. BERT evaluacija

Ako je BERT istreniran, prikazujemo njegove metrike. U suprotnom, potrebno je prvo pokrenuti trening u `04_training.ipynb`.

In [ ]:
if results.get("bert") is not None:
    bert_summary = pd.read_csv(config.RESULTS_DIR / "bert" / "summary_metrics.csv")
    display(bert_summary)
    display(Image(filename=str(config.RESULTS_DIR / "bert" / "confusion_matrix.png")))
else:
    print("BERT checkpoint nije pronađen — evaluacija preskočena.")

## 9. Poređenje LSTM vs. BERT

In [ ]:
comparison_path = config.RESULTS_DIR / "model_comparison.csv"
if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)
    comparison_df
else:
    print("Poređenje nije dostupno — potrebna su oba istrenirana modela.")
    if results.get("lstm") is not None:
        comparison_df = pd.read_csv(config.RESULTS_DIR / "lstm" / "summary_metrics.csv")
        comparison_df.insert(0, "model", "LSTM")
        comparison_df

In [ ]:
if comparison_path.exists():
    plot_df = comparison_df.melt(id_vars="model", var_name="metric", value_name="value")
    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x="metric", y="value", hue="model", palette="Set2")
    plt.title("Poređenje LSTM i BERT modela")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

## Zaključak

Evaluacija je završena. Rezultati su sačuvani u:
- `results/lstm/` — metrike, grafici i primeri grešaka za LSTM
- `results/bert/` — isto za BERT (ako je istreniran)
- `results/model_comparison.csv` — poređenje modela (ako su oba dostupna)